# EMT three-phase inverter state-space extraction

This notebook verifies state-space extraction and modal analysis for a three-phase averaged voltage-source inverter connected to an ideal voltage source through separate R and L components.

Topology:

GND -- Inverter -- nInv -- R -- nMid -- L -- nGrid -- VoltageSource -- GND

The notebook performs three checks:
1. **Check A**: DPsim extracted eigenvalues against an analytical mixed-frame model.
    - This check verifies DPsim's state-space extraction against an analytical model written in the same DPsim-native mixed-frame coordinates.
    - This check includes assertions.  
1. **Check B**: Analytical mixed-frame model against an analytical dq0-frame model.
    - This check shows that the eigenvalues of the native mixed-frame model do not directly match the eigenvalues of the analytical dq0-frame model.
    - It then shows that the analytical mixed-frame model can be transformed into a dq0-frame realization whose eigenvalues match the analytical dq0-frame model.
    - This check uses the analytical mixed-frame model for the transformation because DPsim currently does not expose enough state and frame metadata to transform the extracted model automatically
    - This check has no assertions. 
1. **Check C**: DPsim-native Floquet multipliers against DPsim time-domain simulation.
    - This check demonstrates that, for the native EMT stationary-frame realization, one-period Floquet multipliers provide the relevant stability indicator.
    - It compares DPsim-native Floquet stability indicators with analytical dq0-frame stability indicators and DPsim EMT time-domain simulations.
    - This check has no assertions.

In the native EMT mixed-frame coordinates, the balanced operating point appears as a periodic trajectory. Therefore, instantaneous eigenvalues of one frozen native-frame matrix are not generally valid stability indicators. They are used in Check A only to verify the extracted matrix against an analytical model in the same coordinates. For stability assessment in the native EMT mixed frame, Check C instead uses the Floquet multipliers of the one-period state-transition matrix.

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import dpsimpy

emt = dpsimpy.emt
ph3 = emt.ph3

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType
Solver = dpsimpy.Solver
StateSpaceModalAnalysis = dpsimpy.StateSpaceModalAnalysis

## Parameters

In [12]:
# Simulation settings
time_step = 50e-6
final_time_short = time_step
final_time_validation = 1.0

# System
system_frequency = 50.0
omega0 = 2.0 * np.pi * system_frequency

# Fixed ideal source voltage
grid_voltage_rms_ll = 400.0
source_phase_deg = -90.0

# Grid branch
r_grid = 0.3
l_grid = 1.0e-3

# Inverter filter
lf = 2.0e-3
cf = 10.0e-6
rf = 0.2
rc = 0.2

# PLL
kp_pll = 0.25
ki_pll = 0.2
omega_cutoff = omega0

# Power references
p_ref = 10000.0
q_ref = 5000.0

# Base power controller
kp_power_ctrl_base = 0.05
ki_power_ctrl_base = 0.2

# Base current controller
kp_curr_ctrl_base = 0.25
ki_curr_ctrl_base = 1.0

# Gain scales for the stability check
gain_scale_stable = 3.7
gain_scale_unstable = 3.8

sim_name_base = "EMT_Ph3_Inverter_StateSpaceExtraction"

## Common utilities

In [13]:
abc_scale = np.sqrt(2.0 / 3.0)


def abc_phasor(phase_a_phasor):
    return np.array(
        [
            [phase_a_phasor],
            [phase_a_phasor * np.exp(-1j * 2.0 * np.pi / 3.0)],
            [phase_a_phasor * np.exp(+1j * 2.0 * np.pi / 3.0)],
        ],
        dtype=np.complex128,
    )


def abc_instantaneous_from_phasor(phase_a_phasor, t=0.0):
    return np.real(
        abc_scale * abc_phasor(phase_a_phasor).reshape(3) * np.exp(1j * omega0 * t)
    )


def three_phase_param(value):
    return dpsimpy.Math.single_phase_parameter_to_three_phase(value)


def source_phasor():
    return grid_voltage_rms_ll * np.exp(1j * np.deg2rad(source_phase_deg))


def park_matrix(theta):
    k = np.sqrt(2.0 / 3.0)

    return np.array(
        [
            [
                k * np.cos(theta),
                k * np.cos(theta - 2.0 * np.pi / 3.0),
                k * np.cos(theta + 2.0 * np.pi / 3.0),
            ],
            [
                -k * np.sin(theta),
                -k * np.sin(theta - 2.0 * np.pi / 3.0),
                -k * np.sin(theta + 2.0 * np.pi / 3.0),
            ],
        ],
        dtype=float,
    )


def inv_park_matrix(theta):
    k = np.sqrt(2.0 / 3.0)

    return np.array(
        [
            [k * np.cos(theta), -k * np.sin(theta)],
            [
                k * np.cos(theta - 2.0 * np.pi / 3.0),
                -k * np.sin(theta - 2.0 * np.pi / 3.0),
            ],
            [
                k * np.cos(theta + 2.0 * np.pi / 3.0),
                -k * np.sin(theta + 2.0 * np.pi / 3.0),
            ],
        ],
        dtype=float,
    )


def park_matrix_3(theta):
    k = np.sqrt(2.0 / 3.0)
    k0 = 1.0 / np.sqrt(3.0)

    return np.array(
        [
            [
                k * np.cos(theta),
                k * np.cos(theta - 2.0 * np.pi / 3.0),
                k * np.cos(theta + 2.0 * np.pi / 3.0),
            ],
            [
                -k * np.sin(theta),
                -k * np.sin(theta - 2.0 * np.pi / 3.0),
                -k * np.sin(theta + 2.0 * np.pi / 3.0),
            ],
            [k0, k0, k0],
        ],
        dtype=float,
    )


def numerical_jacobian(f, x0, eps=1e-6):
    x0 = np.asarray(x0, dtype=float)
    f0 = np.asarray(f(x0), dtype=float)
    A = np.zeros((len(f0), len(x0)))

    for k in range(len(x0)):
        h = eps * max(1.0, abs(x0[k]))

        xp = x0.copy()
        xm = x0.copy()

        xp[k] += h
        xm[k] -= h

        A[:, k] = (f(xp) - f(xm)) / (2.0 * h)

    return A


def trapezoidal_discretization(A, dt):
    I = np.eye(A.shape[0])

    return np.linalg.solve(
        I - 0.5 * dt * A,
        I + 0.5 * dt * A,
    )


def bilinear_lambda_from_z(z, dt):
    return (2.0 / dt) * ((z - 1.0) / (z + 1.0))


def finite_complex(values):
    values = np.asarray(values).reshape(-1)

    return values[np.isfinite(values.real) & np.isfinite(values.imag)]


def closest_eigenvalue_matches(reference, candidate):
    rows = []

    for lam_ref in reference:
        idx = np.argmin(np.abs(candidate - lam_ref))
        rows.append(
            {
                "reference": lam_ref,
                "closest": candidate[idx],
                "distance": abs(candidate[idx] - lam_ref),
            }
        )

    return sorted(rows, key=lambda row: row["distance"], reverse=True)


def symmetric_max_eigenvalue_distance(a, b):
    a = finite_complex(a)
    b = finite_complex(b)

    return max(
        closest_eigenvalue_matches(a, b)[0]["distance"],
        closest_eigenvalue_matches(b, a)[0]["distance"],
    )


def eigenvalue_table(lambdas, source, n=None):
    values = finite_complex(lambdas)
    order = np.argsort(values.real)[::-1]

    if n is not None:
        order = order[:n]

    values = values[order]

    return pd.DataFrame(
        {
            "source": source,
            "Re(lambda) [1/s]": values.real,
            "Im(lambda) [rad/s]": values.imag,
            "frequency [Hz]": np.abs(values.imag) / (2.0 * np.pi),
        }
    )

## Shared DPsim system setup

The operating point is computed for a fixed source voltage. The inverter power references are interpreted at the inverter capacitor voltage, not at the external terminal node.

In [14]:
def steady_state_operating_point(gain_scale=1.0):
    kp_power_ctrl = gain_scale * kp_power_ctrl_base
    ki_power_ctrl = gain_scale * ki_power_ctrl_base
    kp_curr_ctrl = gain_scale * kp_curr_ctrl_base
    ki_curr_ctrl = gain_scale * ki_curr_ctrl_base

    S = p_ref + 1j * q_ref
    E = source_phasor()

    Z_total = rc + r_grid + 1j * omega0 * l_grid
    C = Z_total * np.conj(S)

    roots = np.roots(
        [
            1.0,
            -(abs(E) ** 2 + 2.0 * C.real),
            abs(C) ** 2,
        ]
    )

    roots_real = np.real(roots[np.isreal(roots)])
    y = np.max(roots_real[roots_real > 0.0])

    Vc = (y - np.conj(C)) / np.conj(E)
    Iinj = np.conj(S / Vc)

    Uinv = Vc - rc * Iinj
    Umid = E + 1j * omega0 * l_grid * Iinj

    If = Iinj + 1j * omega0 * cf * Vc
    Vref = Vc + (rf + 1j * omega0 * lf) * If

    theta = np.angle(Vc)

    vc_abc = abc_instantaneous_from_phasor(Vc)
    if_abc = abc_instantaneous_from_phasor(If)
    i_line_abc = abc_instantaneous_from_phasor(Iinj)
    u_inv_abc = abc_instantaneous_from_phasor(Uinv)
    u_mid_abc = abc_instantaneous_from_phasor(Umid)
    e_grid_abc = abc_instantaneous_from_phasor(E)
    v_ref_abc = abc_instantaneous_from_phasor(Vref)

    T = park_matrix(theta)

    vc_dq = T @ vc_abc
    i_dq = T @ i_line_abc
    v_ref_dq = T @ v_ref_abc

    vc_d, vc_q = vc_dq
    i_d, i_q = i_dq

    p0 = vc_d * i_d + vc_q * i_q
    q0 = -vc_d * i_q + vc_q * i_d

    phi_pll = 0.0
    p_filtered = p0
    q_filtered = q0

    phi_d = (i_d - kp_power_ctrl * (p_ref - p_filtered)) / ki_power_ctrl
    phi_q = (i_q - kp_power_ctrl * (q_filtered - q_ref)) / ki_power_ctrl

    i_ref_d = kp_power_ctrl * (p_ref - p_filtered) + ki_power_ctrl * phi_d
    i_ref_q = kp_power_ctrl * (q_filtered - q_ref) + ki_power_ctrl * phi_q

    gamma_d = (v_ref_dq[0] - kp_curr_ctrl * (i_ref_d - i_d)) / ki_curr_ctrl
    gamma_q = (v_ref_dq[1] - kp_curr_ctrl * (i_ref_q - i_q)) / ki_curr_ctrl

    x_inv = np.zeros(14)
    x_inv[0] = theta
    x_inv[1] = phi_pll
    x_inv[2] = p_filtered
    x_inv[3] = q_filtered
    x_inv[4] = phi_d
    x_inv[5] = phi_q
    x_inv[6] = gamma_d
    x_inv[7] = gamma_q
    x_inv[8:11] = vc_abc
    x_inv[11:14] = if_abc

    x_full = np.zeros(17)
    x_full[:14] = x_inv
    x_full[14:17] = i_line_abc

    return {
        "E": E,
        "Vc": Vc,
        "Uinv": Uinv,
        "Umid": Umid,
        "Iinj": Iinj,
        "If": If,
        "Vref": Vref,
        "theta": theta,
        "x_inv": x_inv,
        "x_full": x_full,
        "vc_abc": vc_abc,
        "if_abc": if_abc,
        "i_line_abc": i_line_abc,
        "u_inv_abc": u_inv_abc,
        "u_mid_abc": u_mid_abc,
        "e_grid_abc": e_grid_abc,
    }


def build_dpsim_system(op, gain_scale):
    n_inv = emt.SimNode("nInv", PhaseType.ABC)
    n_mid = emt.SimNode("nMid", PhaseType.ABC)
    n_grid = emt.SimNode("nGrid", PhaseType.ABC)

    n_inv.set_initial_voltage(op["Uinv"])
    n_mid.set_initial_voltage(op["Umid"])
    n_grid.set_initial_voltage(op["E"])

    voltage_source = ph3.VoltageSource("VoltageSource")
    voltage_source.set_parameters(
        abc_phasor(op["E"]),
        system_frequency,
    )

    resistor = ph3.Resistor("GridResistor")
    resistor.set_parameters(three_phase_param(r_grid))

    inductor = ph3.Inductor("GridInductor")
    inductor.set_parameters(three_phase_param(l_grid))

    inverter = ph3.AvVoltSourceInverterStateSpace("INV_SSN_PLL")
    inverter.set_parameters(
        lf,
        cf,
        rf,
        rc,
        omega0,
        kp_pll,
        ki_pll,
        omega_cutoff,
        p_ref,
        q_ref,
        gain_scale * kp_power_ctrl_base,
        gain_scale * ki_power_ctrl_base,
        gain_scale * kp_curr_ctrl_base,
        gain_scale * ki_curr_ctrl_base,
    )

    inverter.connect([emt.SimNode.gnd, n_inv])
    resistor.connect([n_inv, n_mid])
    inductor.connect([n_mid, n_grid])
    voltage_source.connect([emt.SimNode.gnd, n_grid])

    system = SystemTopology(
        system_frequency,
        [n_inv, n_mid, n_grid],
        [inverter, resistor, inductor, voltage_source],
    )

    return system, n_inv, n_mid, n_grid, inverter


def get_modal_results(sim):
    extractor = sim.get_state_space_extractor()

    modal = StateSpaceModalAnalysis(extractor)
    modal.update()

    Ad = np.array(extractor.get_discrete_state_matrix(), copy=True)
    z = np.array(modal.get_discrete_eigenvalues(), copy=True).reshape(-1)
    lambdas = np.array(modal.get_continuous_eigenvalues(), copy=True).reshape(-1)

    return {
        "Ad": Ad,
        "z": z,
        "lambda": lambdas,
        "state_count": extractor.get_state_count(),
    }


def run_dpsim(gain_scale, final_time, case_name, log_signals):
    sim_name = f"{sim_name_base}_{case_name}_gain_{gain_scale:g}".replace(".", "_")
    Logger.set_log_dir("logs/" + sim_name)

    op = steady_state_operating_point(gain_scale)
    system, n_inv, n_mid, n_grid, inverter = build_dpsim_system(op, gain_scale)

    sim = Simulation(sim_name)
    sim.set_system(system)

    if log_signals:
        logger = Logger(sim_name)
        logger.log_attribute("p_inst", inverter.attr("p_inst"))
        logger.log_attribute("q_inst", inverter.attr("q_inst"))
        logger.log_attribute("vc_d", inverter.attr("vc_d"))
        logger.log_attribute("vc_q", inverter.attr("vc_q"))
        logger.log_attribute("omega_pll", inverter.attr("omega_pll"))
        sim.add_logger(logger)

    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)

    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)
    sim.do_state_space_extraction(True)

    sim.set_time_step(time_step)
    sim.set_final_time(final_time)

    n_steps = int(round(final_time / time_step))
    modal_first = None

    sim.start()

    for step_idx in range(n_steps):
        sim.next()

        if step_idx == 0:
            modal_first = get_modal_results(sim)

    sim.stop()

    return {
        "sim_name": sim_name,
        "op": op,
        "modal": modal_first,
    }

## Check A: DPsim vs analytical mixed-frame model

This check verifies the DPsim state-space extraction in DPsim's native mixed coordinates.

The analytical model uses the same state coordinates as the extracted DPsim model:

thetaPLL, phiPLL, P, Q, phiD, phiQ, gammaD, gammaQ,
vcA, vcB, vcC, ifA, ifB, ifC,
iLA, iLB, iLC

The comparison is performed using discrete eigenvalues $z$ and continuous eigenvalues $\lambda$.

These instantaneous native-frame eigenvalues are used here only to verify the extracted native EMT state-space realization. They are not used as the stability criterion for the periodic stationary-frame operating point.

In [ ]:
def inverter_rhs_mixed(x_inv, u_abc, gain_scale):
    kp_power_ctrl = gain_scale * kp_power_ctrl_base
    ki_power_ctrl = gain_scale * ki_power_ctrl_base
    kp_curr_ctrl = gain_scale * kp_curr_ctrl_base
    ki_curr_ctrl = gain_scale * ki_curr_ctrl_base

    theta = x_inv[0]
    phi_pll = x_inv[1]
    p_filtered = x_inv[2]
    q_filtered = x_inv[3]
    phi_d = x_inv[4]
    phi_q = x_inv[5]
    gamma_d = x_inv[6]
    gamma_q = x_inv[7]
    vc_abc = x_inv[8:11]
    if_abc = x_inv[11:14]

    T = park_matrix(theta)
    T_inv = inv_park_matrix(theta)

    i_inj_abc = (vc_abc - u_abc) / rc

    vc_dq = T @ vc_abc
    i_dq = T @ i_inj_abc

    vc_d, vc_q = vc_dq
    i_d, i_q = i_dq

    p_inst = vc_d * i_d + vc_q * i_q
    q_inst = -vc_d * i_q + vc_q * i_d

    omega_pll = omega0 + kp_pll * vc_q + ki_pll * phi_pll

    i_ref_d = kp_power_ctrl * (p_ref - p_filtered) + ki_power_ctrl * phi_d
    i_ref_q = kp_power_ctrl * (q_filtered - q_ref) + ki_power_ctrl * phi_q

    v_ref_d = kp_curr_ctrl * (i_ref_d - i_d) + ki_curr_ctrl * gamma_d
    v_ref_q = kp_curr_ctrl * (i_ref_q - i_q) + ki_curr_ctrl * gamma_q

    v_ref_abc = T_inv @ np.array([v_ref_d, v_ref_q])

    dx = np.zeros(14)

    dx[0] = omega_pll
    dx[1] = vc_q
    dx[2] = omega_cutoff * (p_inst - p_filtered)
    dx[3] = omega_cutoff * (q_inst - q_filtered)
    dx[4] = p_ref - p_filtered
    dx[5] = q_filtered - q_ref
    dx[6] = i_ref_d - i_d
    dx[7] = i_ref_q - i_q
    dx[8:11] = if_abc / cf + (u_abc - vc_abc) / (cf * rc)
    dx[11:14] = (v_ref_abc - vc_abc - rf * if_abc) / lf

    return dx


def full_rhs_mixed(x_full, op, gain_scale):
    x_inv = x_full[:14]
    i_line_abc = x_full[14:17]

    vc_abc = x_inv[8:11]
    u_inv_abc = vc_abc - rc * i_line_abc

    dx_inv = inverter_rhs_mixed(x_inv, u_inv_abc, gain_scale)
    di_line = (u_inv_abc - r_grid * i_line_abc - op["e_grid_abc"]) / l_grid

    dx = np.zeros(17)
    dx[:14] = dx_inv
    dx[14:17] = di_line

    return dx


def analytical_mixed_model(op, gain_scale):
    A = numerical_jacobian(
        lambda x: full_rhs_mixed(x, op, gain_scale),
        op["x_full"],
    )

    Ad = trapezoidal_discretization(A, time_step)
    z = np.linalg.eigvals(Ad)
    lambdas = bilinear_lambda_from_z(z, time_step)

    return {
        "A": A,
        "Ad": Ad,
        "z": z,
        "lambda": lambdas,
    }


gain_scale_check_a = gain_scale_stable

dpsim_check_a = run_dpsim(
    gain_scale=gain_scale_check_a,
    final_time=final_time_short,
    case_name="check_a",
    log_signals=False,
)

op_check_a = dpsim_check_a["op"]
modal_dpsim_check_a = dpsim_check_a["modal"]
analytic_mixed_check_a = analytical_mixed_model(op_check_a, gain_scale_check_a)

z_error_check_a = symmetric_max_eigenvalue_distance(
    modal_dpsim_check_a["z"],
    analytic_mixed_check_a["z"],
)

lambda_error_check_a = symmetric_max_eigenvalue_distance(
    modal_dpsim_check_a["lambda"],
    analytic_mixed_check_a["lambda"],
)

print("Check A: DPsim extracted eigenvalues vs analytical mixed-frame eigenvalues")
print("DPsim state count:", modal_dpsim_check_a["state_count"])
print("Analytical mixed-frame state count:", analytic_mixed_check_a["Ad"].shape[0])
print("Maximum discrete eigenvalue mismatch |Δz|:", z_error_check_a)
print("Maximum continuous eigenvalue mismatch |Δlambda|:", lambda_error_check_a)

assert modal_dpsim_check_a["state_count"] == analytic_mixed_check_a["Ad"].shape[0]
assert z_error_check_a < 1e-8
assert lambda_error_check_a < 1e-4

print("Check A passed.")

In [ ]:
print("Check A rightmost eigenvalues sorted by decreasing Re(lambda):")

pd.concat(
    [
        eigenvalue_table(
            modal_dpsim_check_a["lambda"],
            "DPsim extracted",
            n=8,
        ),
        eigenvalue_table(
            analytic_mixed_check_a["lambda"],
            "Analytical mixed-frame",
            n=8,
        ),
    ],
    ignore_index=True,
)

## Check B: analytical mixed frame vs analytical dq0 frame

This section explains the relation between DPsim's native mixed-frame eigenvalues and a complete dq0-frame model.

First, the DPsim-native eigenvalues are compared with direct analytical dq0 eigenvalues. They are not expected to match directly because they are expressed in different frame realizations.

Then, the analytical mixed-frame model is transformed to dq0 using

A_dq0 = S A_mixed S^{-1} + Sdot S^{-1}

and compared with the direct analytical dq0 model.

This transformation is performed using the analytical mixed-frame model because state-space model extracted from DPsim currently does not expose enough state and frame metadata to build the transformation automatically.

In [ ]:
def source_dq0(delta):
    E = grid_voltage_rms_ll

    return np.array(
        [
            E * np.cos(delta),
            -E * np.sin(delta),
            0.0,
        ],
        dtype=float,
    )


def dq0_operating_point(op):
    theta_pll = op["x_full"][0]
    theta_grid = np.angle(op["E"])

    T3 = park_matrix_3(theta_pll)

    vc_dq0 = T3 @ op["x_full"][8:11]
    if_dq0 = T3 @ op["x_full"][11:14]
    il_dq0 = T3 @ op["x_full"][14:17]

    x = np.zeros(17)

    x[0:8] = op["x_full"][0:8]
    x[0] = theta_pll - theta_grid

    x[8:10] = vc_dq0[:2]
    x[10:12] = if_dq0[:2]
    x[12:14] = il_dq0[:2]

    x[14] = vc_dq0[2]
    x[15] = if_dq0[2]
    x[16] = il_dq0[2]

    return x


def full_rhs_dq0(x, gain_scale):
    kp_power_ctrl = gain_scale * kp_power_ctrl_base
    ki_power_ctrl = gain_scale * ki_power_ctrl_base
    kp_curr_ctrl = gain_scale * kp_curr_ctrl_base
    ki_curr_ctrl = gain_scale * ki_curr_ctrl_base

    J = np.array(
        [
            [0.0, 1.0],
            [-1.0, 0.0],
        ]
    )

    delta = x[0]
    phi_pll = x[1]
    p_filtered = x[2]
    q_filtered = x[3]
    phi_d = x[4]
    phi_q = x[5]
    gamma_d = x[6]
    gamma_q = x[7]

    vc = x[8:10]
    ifilt = x[10:12]
    iline = x[12:14]

    vc0 = x[14]
    if0 = x[15]
    iline0 = x[16]

    vc_d, vc_q = vc
    i_d, i_q = iline

    e = source_dq0(delta)[:2]

    p_inst = vc_d * i_d + vc_q * i_q
    q_inst = -vc_d * i_q + vc_q * i_d

    omega_pll = omega0 + kp_pll * vc_q + ki_pll * phi_pll

    i_ref_d = kp_power_ctrl * (p_ref - p_filtered) + ki_power_ctrl * phi_d
    i_ref_q = kp_power_ctrl * (q_filtered - q_ref) + ki_power_ctrl * phi_q

    v_ref = np.array(
        [
            kp_curr_ctrl * (i_ref_d - i_d) + ki_curr_ctrl * gamma_d,
            kp_curr_ctrl * (i_ref_q - i_q) + ki_curr_ctrl * gamma_q,
        ]
    )

    dx = np.zeros(17)

    dx[0] = omega_pll - omega0
    dx[1] = vc_q
    dx[2] = omega_cutoff * (p_inst - p_filtered)
    dx[3] = omega_cutoff * (q_inst - q_filtered)
    dx[4] = p_ref - p_filtered
    dx[5] = q_filtered - q_ref
    dx[6] = i_ref_d - i_d
    dx[7] = i_ref_q - i_q

    dx[8:10] = omega_pll * (J @ vc) + ifilt / cf - iline / cf
    dx[10:12] = omega_pll * (J @ ifilt) + (v_ref - vc - rf * ifilt) / lf
    dx[12:14] = omega_pll * (J @ iline) + (vc - (rc + r_grid) * iline - e) / l_grid

    dx[14] = if0 / cf - iline0 / cf
    dx[15] = (-vc0 - rf * if0) / lf
    dx[16] = (vc0 - (rc + r_grid) * iline0) / l_grid

    return dx


def analytical_dq0_model(op, gain_scale):
    x0 = dq0_operating_point(op)

    A = numerical_jacobian(
        lambda x: full_rhs_dq0(x, gain_scale),
        x0,
    )

    return {
        "x0": x0,
        "A": A,
        "lambda": np.linalg.eigvals(A),
    }


def build_native_to_dq0_transform(x0_native):
    theta = x0_native[0]
    vc_abc = x0_native[8:11]
    if_abc = x0_native[11:14]
    il_abc = x0_native[14:17]

    T3 = park_matrix_3(theta)

    tD = T3[0, :]
    tQ = T3[1, :]
    t0 = T3[2, :]

    d_tD = tQ
    d_tQ = -tD

    S = np.zeros((17, 17))

    S[0:8, 0:8] = np.eye(8)

    S[8, 0] = d_tD @ vc_abc
    S[8, 8:11] = tD
    S[9, 0] = d_tQ @ vc_abc
    S[9, 8:11] = tQ

    S[10, 0] = d_tD @ if_abc
    S[10, 11:14] = tD
    S[11, 0] = d_tQ @ if_abc
    S[11, 11:14] = tQ

    S[12, 0] = d_tD @ il_abc
    S[12, 14:17] = tD
    S[13, 0] = d_tQ @ il_abc
    S[13, 14:17] = tQ

    S[14, 8:11] = t0
    S[15, 11:14] = t0
    S[16, 14:17] = t0

    return S


def transformed_dq0_matrix_from_mixed(op, A_mixed):
    x0 = op["x_full"]
    theta0 = x0[0]
    h = 1e-6

    T0 = park_matrix_3(theta0)
    dq0_vc = T0 @ x0[8:11]
    dq0_if = T0 @ x0[11:14]
    dq0_il = T0 @ x0[14:17]

    x_minus = x0.copy()
    x_plus = x0.copy()

    x_minus[0] = theta0 - h
    x_plus[0] = theta0 + h

    x_minus[8:11] = park_matrix_3(theta0 - h).T @ dq0_vc
    x_minus[11:14] = park_matrix_3(theta0 - h).T @ dq0_if
    x_minus[14:17] = park_matrix_3(theta0 - h).T @ dq0_il

    x_plus[8:11] = park_matrix_3(theta0 + h).T @ dq0_vc
    x_plus[11:14] = park_matrix_3(theta0 + h).T @ dq0_if
    x_plus[14:17] = park_matrix_3(theta0 + h).T @ dq0_il

    S_minus = build_native_to_dq0_transform(x_minus)
    S0 = build_native_to_dq0_transform(x0)
    S_plus = build_native_to_dq0_transform(x_plus)

    dS_dtheta = (S_plus - S_minus) / (2.0 * h)
    Sdot = omega0 * dS_dtheta

    return S0 @ A_mixed @ np.linalg.inv(S0) + Sdot @ np.linalg.inv(S0)


dq0_direct = analytical_dq0_model(op_check_a, gain_scale_check_a)

A_dq0_from_mixed = transformed_dq0_matrix_from_mixed(
    op_check_a,
    analytic_mixed_check_a["A"],
)

lambda_dq0_direct = finite_complex(dq0_direct["lambda"])
lambda_dq0_from_mixed = finite_complex(np.linalg.eigvals(A_dq0_from_mixed))

dpsim_vs_dq0_error = symmetric_max_eigenvalue_distance(
    modal_dpsim_check_a["lambda"],
    lambda_dq0_direct,
)

dq0_consistency_error = symmetric_max_eigenvalue_distance(
    lambda_dq0_direct,
    lambda_dq0_from_mixed,
)

print("Check B.1: DPsim native eigenvalues vs direct analytical dq0 eigenvalues")
print("Maximum nearest-neighbor difference:", dpsim_vs_dq0_error)
print("Expected result: not zero, because the frames are different.")

print()
print(
    "Check B.2: direct analytical dq0 eigenvalues vs transformed analytical mixed-frame eigenvalues"
)
print("Direct analytical dq0 state count:", dq0_direct["A"].shape[0])
print("Transformed analytical mixed-frame state count:", A_dq0_from_mixed.shape[0])
print("Maximum dq0 eigenvalue mismatch:", dq0_consistency_error)

In [ ]:
print("Check B rightmost eigenvalues sorted by decreasing Re(lambda):")

pd.concat(
    [
        eigenvalue_table(
            modal_dpsim_check_a["lambda"],
            "DPsim native",
            n=8,
        ),
        eigenvalue_table(
            lambda_dq0_direct,
            "Analytical dq0",
            n=8,
        ),
        eigenvalue_table(
            lambda_dq0_from_mixed,
            "Analytical mixed transformed to dq0",
            n=8,
        ),
    ],
    ignore_index=True,
)

## Check C: DPsim-native Floquet multipliers vs DPsim time-domain simulation

For the native EMT stationary-frame realization, the balanced operating point is periodic. Therefore, instantaneous eigenvalues of a single frozen native-frame matrix are not generally valid stability indicators.

This check uses the one-period monodromy matrix

Phi = Ad[N-1] ... Ad[1] Ad[0]

and the Floquet multipliers

mu_i = eig(Phi)

as the native-frame stability indicator.

As a reference, the same operating point is analyzed with the analytical dq0-frame model. Since the balanced operating point is an equilibrium in this frame, its continuous-time eigenvalues can be used with the usual left-half-plane stability criterion. The comparison is shown for two controller-gain scales close to the stability boundary.

In [ ]:
def load_log(sim_name):
    return pd.read_csv(
        Path("logs") / sim_name / f"{sim_name}.csv",
        skipinitialspace=True,
    )


def signal(df, name):
    cols = [c for c in df.columns if c == name or c.startswith(name + "_")]

    return df[cols].to_numpy()


def scalar_signal(df, name):
    return signal(df, name)[:, 0]


def time_vector(df):
    return df.iloc[:, 0].to_numpy()


def plot_floquet_multipliers(ax, mu, title):
    values = finite_complex(mu)

    phi = np.linspace(0.0, 2.0 * np.pi, 500)

    ax.plot(
        np.cos(phi),
        np.sin(phi),
        linestyle="--",
        linewidth=0.8,
        label="unit circle",
    )

    ax.scatter(values.real, values.imag, s=18)

    ax.axhline(0.0, linewidth=0.8)
    ax.axvline(0.0, linewidth=0.8)

    ax.set_xlabel("Re(μ)")
    ax.set_ylabel("Im(μ)")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True)
    ax.legend()


def plot_continuous_eigenvalues(ax, lambdas, title):
    values = finite_complex(lambdas)

    ax.scatter(values.real, values.imag, s=18)

    ax.axhline(0.0, linewidth=0.8)
    ax.axvline(0.0, linewidth=0.8)

    ax.set_xlabel("Re(λ) [1/s]")
    ax.set_ylabel("Im(λ) [rad/s]")
    ax.set_title(title)
    ax.grid(True)


def plot_active_power_response(ax, df, title):
    t = time_vector(df)
    p = scalar_signal(df, "p_inst")

    (p_line,) = ax.plot(t, p, label="DPsim p_inst")

    ax.axhline(
        p_ref,
        linestyle="--",
        color=p_line.get_color(),
        label="p_ref",
    )

    ax.set_ylim(p_ref * 0.99, p_ref * 1.01)

    ax.set_xlabel("time [s]")
    ax.set_ylabel("active power [W]")
    ax.set_title(title)
    ax.grid(True)
    ax.legend()


def run_dpsim_collect_ad_over_period(gain_scale, case_name, skip_periods=1):
    period = 1.0 / system_frequency
    steps_per_period = int(round(period / time_step))
    total_steps = (skip_periods + 1) * steps_per_period

    sim_name = f"{sim_name_base}_{case_name}_floquet_gain_{gain_scale:g}".replace(
        ".", "_"
    )
    Logger.set_log_dir("logs/" + sim_name)

    op = steady_state_operating_point(gain_scale)
    system, n_inv, n_mid, n_grid, inverter = build_dpsim_system(op, gain_scale)

    sim = Simulation(sim_name)
    sim.set_system(system)

    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)

    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)
    sim.do_state_space_extraction(True)

    sim.set_time_step(time_step)
    sim.set_final_time(total_steps * time_step)

    ad_over_period = []

    sim.start()

    for step_idx in range(total_steps):
        sim.next()

        if step_idx >= skip_periods * steps_per_period:
            ad_over_period.append(get_modal_results(sim)["Ad"])

    sim.stop()

    return {
        "op": op,
        "ad_over_period": ad_over_period,
        "period": steps_per_period * time_step,
        "steps_per_period": steps_per_period,
    }


def floquet_from_ad_sequence(ad_sequence, period):
    Phi = np.eye(ad_sequence[0].shape[0])

    for Ad in ad_sequence:
        Phi = Ad @ Phi

    mu = np.linalg.eigvals(Phi)
    growth_rates = np.log(np.abs(mu)) / period

    return {
        "Phi": Phi,
        "mu": mu,
        "growth_rates": growth_rates,
    }


def stability_summary(gain_scale):
    native_periodic = run_dpsim_collect_ad_over_period(
        gain_scale=gain_scale,
        case_name="check_c",
        skip_periods=1,
    )

    floquet_native = floquet_from_ad_sequence(
        native_periodic["ad_over_period"],
        native_periodic["period"],
    )

    dq0 = analytical_dq0_model(
        native_periodic["op"],
        gain_scale,
    )

    lambda_dq0 = finite_complex(dq0["lambda"])

    return {
        "gain_scale": gain_scale,
        "steps_per_period": native_periodic["steps_per_period"],
        "mu": finite_complex(floquet_native["mu"]),
        "lambda_dq0": lambda_dq0,
        "max_abs_mu": np.max(np.abs(floquet_native["mu"])),
        "max_floquet_growth_rate": np.max(floquet_native["growth_rates"]),
        "max_re_lambda_dq0": np.max(lambda_dq0.real),
    }


def run_dpsim_time_domain(gain_scale, case_name):
    return run_dpsim(
        gain_scale=gain_scale,
        final_time=final_time_validation,
        case_name=case_name,
        log_signals=True,
    )


summary_stable = stability_summary(gain_scale_stable)
summary_unstable = stability_summary(gain_scale_unstable)

dpsim_stable = run_dpsim_time_domain(
    gain_scale=gain_scale_stable,
    case_name="check_c_stable",
)

dpsim_unstable = run_dpsim_time_domain(
    gain_scale=gain_scale_unstable,
    case_name="check_c_unstable",
)

df_stable = load_log(dpsim_stable["sim_name"])
df_unstable = load_log(dpsim_unstable["sim_name"])

print("Check C: DPsim-native Floquet multipliers vs analytical dq0 eigenvalues")
print("Steps per period:", summary_stable["steps_per_period"])

pd.DataFrame(
    [
        {
            "case": "stable",
            "gain scale": summary_stable["gain_scale"],
            "max |mu|, DPsim-native Floquet": summary_stable["max_abs_mu"],
            "max Floquet growth rate [1/s]": summary_stable["max_floquet_growth_rate"],
            "max Re(lambda_dq0) [1/s]": summary_stable["max_re_lambda_dq0"],
        },
        {
            "case": "unstable",
            "gain scale": summary_unstable["gain_scale"],
            "max |mu|, DPsim-native Floquet": summary_unstable["max_abs_mu"],
            "max Floquet growth rate [1/s]": summary_unstable[
                "max_floquet_growth_rate"
            ],
            "max Re(lambda_dq0) [1/s]": summary_unstable["max_re_lambda_dq0"],
        },
    ]
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

plot_floquet_multipliers(
    axes[0, 0],
    summary_stable["mu"],
    f"DPsim-native Floquet multipliers, gain = {gain_scale_stable:g}",
)

plot_continuous_eigenvalues(
    axes[0, 1],
    summary_stable["lambda_dq0"],
    f"Analytical dq0 eigenvalues, gain = {gain_scale_stable:g}",
)

plot_active_power_response(
    axes[0, 2],
    df_stable,
    f"DPsim EMT simulation, gain = {gain_scale_stable:g}",
)

plot_floquet_multipliers(
    axes[1, 0],
    summary_unstable["mu"],
    f"DPsim-native Floquet multipliers, gain = {gain_scale_unstable:g}",
)

plot_continuous_eigenvalues(
    axes[1, 1],
    summary_unstable["lambda_dq0"],
    f"Analytical dq0 eigenvalues, gain = {gain_scale_unstable:g}",
)

plot_active_power_response(
    axes[1, 2],
    df_unstable,
    f"DPsim EMT simulation, gain = {gain_scale_unstable:g}",
)

plt.tight_layout()
plt.show()